# Phase 4B | Frozen-model, multi-seed computational evaluation and manuscript pack

**Research purpose:** consolidate Phase 4A **observed mouse neural forecasting** and run a larger **counterfactual control simulation** without retraining or silently changing the GRU/Hybrid GRU architectures.

> **Strict evidence separation:** `REAL_*` reflects saved Phase 4A predictions from DANDI 000568 (two sessions of **one** mouse); `SIMULATION_ONLY_*` is hypothetical and is not observed optogenetic efficacy. No human epilepsy treatment, safe intervention, independent-animal generalization, or experimentally calibrated optical response is established.

**No Kaggle Input required.** A small signed-by-hash evidence bundle from the user's actual Phase 4A output is embedded below. Kaggle **Internet ON**, **Accelerator None / CPU**. Two pinned small NWB files (~73 MB combined) are downloaded solely to re-create observed held-out starting histories and verify the frozen model weights.

**Predeclared outcomes:** forecasting MAE/RMSE/R²; 90% interval coverage in all vs high-target states; controller mean rate, excess above target, action usage, harm frequency vs NO_LIGHT, zero-effect invariance, sign reversals, and robustness across two simulators. All intervals over synthetic seeds are descriptive; seeds are **not** biological replicates.

## 1. Frozen evidence package and reproducible setup

The bundled checkpoints and summary tables are read from the user's completed **Phase 4A results ZIP**. No new training occurs here. Full-mode simulations increase *synthetic* replication only.

In [1]:
import os, io, re, math, hashlib, zipfile, base64, json, random, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import h5py, requests, torch
from torch import nn
from scipy.ndimage import binary_dilation
from sklearn.metrics import r2_score

SEED=20261014
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
QUICK_MODE=True                   # True first; False for final computational results
SMOKE_MODE=os.environ.get('PHASE4B_SMOKE','0')=='1'  # Developer-only synthetic test
SIM_SEEDS=6 if QUICK_MODE else 20
SIM_STARTS=3 if QUICK_MODE else 6
SIM_STEPS=35 if QUICK_MODE else 60
BOOT_REPS=250 if QUICK_MODE else 1000
BIN_SEC=0.1; HISTORY_BINS=20
HORIZONS=np.array([1,10,30,60],dtype=int)
HORIZON_NAMES=['0.1s','1s','3s','6s']
STRIDE_BINS=10; MAX_H=60; GUARD_SEC=.3
DANDISET='000568'; VERSION='0.230705.1633'
BASE='https://api.dandiarchive.org'
ROOT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT=ROOT/'phase4b_outputs'; CACHE=ROOT/'phase4b_nwb_cache'
OUT.mkdir(exist_ok=True,parents=True); CACHE.mkdir(exist_ok=True,parents=True)
SIM_PLANTS=['AR1','REPLAY']
SCENARIOS=['INHIBIT_STRONG','INHIBIT_WEAK','ZERO_EFFECT','EXCITATORY','SIGN_REVERSAL','SLOW_ACTUATOR']
POLICIES=['NO_LIGHT','FIXED','THRESHOLD_HALF','THRESHOLD','MPC_NOMINAL','MPC_UNCERTAINTY_GATE','MPC_SIGN_AMBIGUOUS']
print('QUICK_MODE=',QUICK_MODE,'SIM_SEEDS=',SIM_SEEDS,'STARTS=',SIM_STARTS,'STEPS=',SIM_STEPS)
print('No model training; CPU simulation. Synthetic replicates are not animal replicates.')

QUICK_MODE= True SIM_SEEDS= 6 STARTS= 3 STEPS= 35
No model training; CPU simulation. Synthetic replicates are not animal replicates.


## 2. Recover original Phase 4A outputs, and verify exact model/evidence provenance

The archived checkpoints and saved predictions are **real-source artifacts**, not simulation fixtures. Exact source identifiers and checksums are exported. If the source does not match, the notebook stops rather than silently substituting predictions.

In [2]:
BUNDLE_B64 = """..."""
EXPECTED_BUNDLE_SHA256='8e2a5eb5be42bce9e2e3465918222548547ffac7d53e87b7411f9fd9b0fc0fed'
raw=base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(raw).hexdigest()==EXPECTED_BUNDLE_SHA256, 'Phase 4A evidence bundle corrupted'
with zipfile.ZipFile(io.BytesIO(raw)) as zi:
    for n in zi.namelist():
        assert '/' not in n and '..' not in n
        (OUT/n).write_bytes(zi.read(n))

def csv(name):return pd.read_csv(OUT/name)
pr=csv('REAL_heldout_predictions.csv')
original_metrics=csv('REAL_heldout_forecast_metrics.csv')
ci=csv('REAL_interval_coverage_and_width.csv')
roll=csv('REAL_free_running_6s_metrics.csv')
limits=csv('EVIDENCE_GATES_AND_LIMITATIONS.csv')
thresholds=csv('REAL_training_thresholds_and_leaky_params.csv')
calquant=csv('REAL_calibration_quantiles.csv')
source=json.loads((OUT/'PROVENANCE_AND_LIMITATIONS.json').read_text())
assert source['dandiset']==DANDISET and source['pinned_version']==VERSION
assert source['biological_animals']==1 and set(source['sessions'])=={'session_9','session_12'}
assert set(pr['test_session'])==set(source['sessions'])
assert len(pr)==4400, f'Expected 2 sessions x 4 horizons x 550 predictions, got {len(pr)}'
assert {'GRU_uniform_pred_Hz','Hybrid_GRU_adaptive_rollout_pred_Hz','observed_Hz'}.issubset(pr.columns)
print('Phase 4A source observations (same mouse):',len(pr))
print('Frozen checkpoint files:',len(list(OUT.glob('REAL_fold*_member*.pt'))))
print('Artifact SHA256:',EXPECTED_BUNDLE_SHA256)

Phase 4A source observations (same mouse): 4400
Frozen checkpoint files: 8
Artifact SHA256: 8e2a5eb5be42bce9e2e3465918222548547ffac7d53e87b7411f9fd9b0fc0fed


## 3. Recalculate the observed prediction metrics and uncertainty diagnostics (NO retraining)

Metrics are recomputed from the saved observed targets and saved GRU/Hybrid-GRU predictions. A high-activity subgroup uses the **future observed outcome** only for *retrospective evaluation*, never an online controller. Intervals originally calibrated in Phase 4A remain under session shift and lack clinical coverage guarantees.

In [3]:
rows=[]
for (sess,h),grp in pr.groupby(['test_session','horizon'],sort=True):
    y=grp.observed_Hz.to_numpy(dtype=float)
    for model in ['GRU_uniform','Hybrid_GRU_adaptive_rollout']:
        p=grp[model+'_pred_Hz'].to_numpy(dtype=float)
        d=p-y
        r2=float(r2_score(y,p))
        rows.append({'dataset':'DANDI000568','independent_mice':1,'heldout_session':sess,'horizon':h,
                     'model':model,'N_windows':len(grp),'MAE_Hz':np.mean(abs(d)),
                     'RMSE_Hz':np.sqrt(np.mean(d**2)),'MSE_Hz2':np.mean(d**2),
                     'R2':r2,'Bias_Hz':np.mean(d),'P90_AE_Hz':np.quantile(abs(d),.90)})
actual=pd.DataFrame(rows)
reported=original_metrics.rename(columns={'test_session':'heldout_session'})
check=actual.merge(reported,on=['heldout_session','horizon','model'],suffixes=('_new','_saved'),validate='one_to_one')
assert len(check)==16
assert np.allclose(check.MAE_Hz_new,check.MAE_Hz_saved,rtol=0,atol=1e-8), 'Frozen metrics mismatch'
assert np.allclose(check.RMSE_Hz_new,check.RMSE_Hz_saved,rtol=0,atol=1e-8)
actual.to_csv(OUT/'REAL_Table1_Frozen_Heldout_Regression.csv',index=False)
ci[ci.model.isin(['GRU_uniform','Hybrid_GRU_adaptive_rollout'])].to_csv(
    OUT/'REAL_Table2_Uncertainty_Coverage_Observed.csv',index=False)
roll.to_csv(OUT/'REAL_Table3_FreeRunning_Observed.csv',index=False)
print(actual[['heldout_session','horizon','model','MAE_Hz','R2']].round(3).to_string(index=False))
print('Saved real-data error scores independently matched.')

heldout_session horizon                       model  MAE_Hz     R2
     session_12    0.1s                 GRU_uniform   0.962  0.172
     session_12    0.1s Hybrid_GRU_adaptive_rollout   0.980  0.160
     session_12      1s                 GRU_uniform   1.149 -0.039
     session_12      1s Hybrid_GRU_adaptive_rollout   1.140 -0.014
     session_12      3s                 GRU_uniform   1.086 -0.024
     session_12      3s Hybrid_GRU_adaptive_rollout   1.078  0.011
     session_12      6s                 GRU_uniform   1.143 -0.087
     session_12      6s Hybrid_GRU_adaptive_rollout   1.132 -0.033
      session_9    0.1s                 GRU_uniform   0.756  0.238
      session_9    0.1s Hybrid_GRU_adaptive_rollout   0.780  0.213
      session_9      1s                 GRU_uniform   0.802  0.052
      session_9      1s Hybrid_GRU_adaptive_rollout   0.809  0.065
      session_9      3s                 GRU_uniform   0.885  0.014
      session_9      3s Hybrid_GRU_adaptive_rollout   0.883  0

## 4. Recreate held-out neural histories from the exact pinned DANDI version

The real recordings are used to initialize simulation episodes and verify that saved **frozen GRU checkpoints** reproduce the Phase 4A predictions. They do **not** supply treatment-response labels. When `SMOKE_MODE` is set by a developer, synthetic fixtures replace downloads and publication exports are disabled.

In [4]:
if SMOKE_MODE:
    def synth(seed):
        r=np.random.default_rng(seed); n=45000
        a=np.zeros(n,dtype=np.float32);a[0]=2.
        z=r.normal(0,.5,n)
        for i in range(1,n): a[i]=max(0,2+.52*(a[i-1]-2)+z[i]+(1.0 if i%1000<12 else 0))
        return {'session':f'session_{seed}','rate':a,'near_light':np.zeros(n,dtype=bool),'n_bins':n}
    sessions={'session_9':synth(9),'session_12':synth(12)}
    print('!!! SYNTHETIC SMOKE TEST - DO NOT USE OUTPUTS FOR PUBLICATION !!!')
else:
    assets=[];url=f'{BASE}/api/dandisets/{DANDISET}/versions/{VERSION}/assets/'
    while url:
        reply=requests.get(url,timeout=(20,90));reply.raise_for_status()
        d=reply.json();assets+=d['results'];url=d.get('next')
    manifest=pd.DataFrame([{'asset_id':a['asset_id'],'path':a['path'],'size_bytes':int(a['size'])}
                           for a in assets])
    selected=manifest[manifest.path.str.endswith('.nwb')&
                      manifest.path.str.contains('+ogen',regex=False)&
                      manifest.path.str.contains('no-raw-data',regex=False)&
                      (manifest.size_bytes<70_000_000)].sort_values('size_bytes')
    assert len(selected)==2,'Expected two pinned small NWB sessions'
    selected.to_csv(OUT/'REAL_manifest_pinned.csv',index=False)
    files={}
    for _,row in selected.iterrows():
        sid=re.search(r'sess(\d+)',row['path']);assert sid
        path=CACHE/Path(row['path']).name
        if not path.exists() or path.stat().st_size!=int(row.size_bytes):
            tmp=path.with_suffix('.part')
            with requests.get(f"{BASE}/api/assets/{row.asset_id}/download/",stream=True,timeout=(30,180)) as res:
                res.raise_for_status()
                with tmp.open('wb') as fh:
                    for piece in res.iter_content(chunk_size=1_048_576):
                        if piece:fh.write(piece)
            assert tmp.stat().st_size==int(row.size_bytes),'Downloaded NWB size mismatch'
            tmp.replace(path)
        files[f'session_{sid.group(1)}']=path
    assert set(files)=={'session_9','session_12'}
    def optical_mask(hf,n):
        root=hf.get('/stimulus/presentation'); assert root is not None
        delta=np.zeros(n+1,np.int32)
        found=0
        for name,g in root.items():
            if not isinstance(g,h5py.Group) or not name.startswith('OptogeneticSeriesSite'):continue
            if 'data' not in g or 'timestamps' not in g:continue
            times=np.asarray(g['timestamps'][:],dtype=float)
            vals=np.asarray(g['data'][:],dtype=float).reshape(-1)
            if len(times)!=len(vals) or len(times)<3 or np.any(np.diff(times)<0):continue
            on=np.isfinite(vals)&(vals>0)
            starts=np.flatnonzero(on&~np.r_[False,on[:-1]])
            stops=np.flatnonzero(~on&np.r_[False,on[:-1]])
            if on[0]:starts=starts[1:]
            inds=np.searchsorted(stops,starts,side='right')
            for s,i in zip(starts,inds):
                if i>=len(stops):continue
                e=stops[i]
                if not (0<=times[s]<times[e]):continue
                a=int(np.floor(times[s]/BIN_SEC))
                b=int(np.floor(np.nextafter(times[e],-np.inf)/BIN_SEC))
                if 0<=a<=b<n:
                    delta[a]+=1;delta[b+1]-=1;found+=1
        assert found>100,'No aligned light commands'
        mask=np.cumsum(delta[:-1])>0
        radius=max(1,int(math.ceil(GUARD_SEC/BIN_SEC)))
        return binary_dilation(mask,structure=np.ones(2*radius+1,dtype=bool)),found
    sessions={}
    for name,p in files.items():
        with h5py.File(p,'r') as hf:
            spikes=np.asarray(hf['/units/spike_times'][:],dtype=np.float64)
            nu=len(hf['/units/spike_times_index'])
            spikes=spikes[np.isfinite(spikes)&(spikes>=0)]
            # End time matched Phase 4A (including last optical offset).
            latest=float(spikes.max())
            presentation=hf['/stimulus/presentation']
            for g in presentation.values():
                if isinstance(g,h5py.Group) and 'timestamps' in g:
                    t=np.asarray(g['timestamps'][:],dtype=float)
                    if len(t):latest=max(latest,float(np.nanmax(t)))
            n=int(np.ceil(latest/BIN_SEC))+1
            counts=np.bincount(np.floor(spikes/BIN_SEC).astype(np.int64),minlength=n)
            rate=np.asarray(counts/(nu*BIN_SEC),dtype=np.float32)
            near, nev=optical_mask(hf,n)
        sessions[name]={'session':name,'rate':rate,'near_light':near,'n_bins':n}
        print(name,'units=',nu,'bins=',n,'optical events=',nev,'light-free fraction=',round(float(np.mean(~near)),3))

session_12 units= 47 bins= 148103 optical events= 50309 light-free fraction= 0.55
session_9 units= 57 bins= 179202 optical events= 42088 light-free fraction= 0.759


## 5. Load Phase 4A frozen GRU weights, and run an exact-weight verification

This is an **immutable inference model**, not newly trained. Reconstruct its preprocessing from the checkpoint itself. No optical intensity codes are used as physical watts.

In [5]:
class FrozenMultiHorizonGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.GRU(input_size=1,hidden_size=32,batch_first=True)
        self.head=nn.Sequential(nn.Linear(32,32),nn.Tanh(),nn.Linear(32,4))
    def forward(self,x):
        _,h=self.rnn(x)
        return self.head(h[-1])

models=[];scalers=[];pars=[]
for m in [1,2]:
    ckpt=torch.load(OUT/f'REAL_fold1_GRU_uniform_member{m}.pt',map_location='cpu',weights_only=True)
    assert ckpt['kind']=='GRU_uniform' and ckpt['train_session']=='session_9'
    model=FrozenMultiHorizonGRU().eval();model.load_state_dict(ckpt['state_dict'])
    models.append(model);scalers.append(ckpt['scaler']);pars.append(ckpt['params'])
assert scalers[0]==scalers[1]
sc=scalers[0];par=pars[0];mu=float(par['mu_Hz']);alpha=float(par['alpha'])

@torch.no_grad()
def frozen_predict(hist):
    h=np.asarray(hist[-HISTORY_BINS:],dtype=np.float32)
    assert h.shape==(HISTORY_BINS,) and np.isfinite(h).all()
    x=((np.log1p(np.maximum(h,0))-sc['x_mu'])/sc['x_sd']).reshape(1,HISTORY_BINS,1)
    xx=torch.from_numpy(x.astype(np.float32))
    pp=[]
    for model in models:
        z=model(xx).numpy()[0]
        logp=z*np.asarray(sc['y_sd'])+np.asarray(sc['y_mu'])
        yp=np.expm1(np.clip(logp,0,np.log1p(500.)))
        pp.append(float(yp[0]))
    pp=np.asarray(pp)
    return float(pp.mean()),float(pp.std(ddof=1))

if not SMOKE_MODE:
    saved=pr[(pr.test_session=='session_12')&(pr.horizon=='0.1s')].sample(12,random_state=71)
    errs=[];s=sessions['session_12']
    for row in saved.itertuples():
        anchor=int(round(row.t_sec/BIN_SEC))
        history=s['rate'][anchor-HISTORY_BINS:anchor]
        pred,spread=frozen_predict(history)
        errs.append(abs(pred-row.GRU_uniform_pred_Hz))
    print('Frozen checkpoint vs saved Phase4A prediction: worst absolute difference',max(errs))
    assert max(errs)<0.003,'Phase4A checkpoint prediction not reproduced; abort'
else:print('Checkpoint consistency with real held-out recording intentionally NOT tested in smoke mode')

Frozen checkpoint vs saved Phase4A prediction: worst absolute difference 2.070512339358288e-07


## 6. Reconstruct original calibration window for MPC uncertainty gate

Uncertainty thresholds use only the *training-session's* calibration period. No held-out-session outcome tunes a decision. As in Phase 4A, this gate is a **heuristic**, not a safety certificate.

In [6]:
def sample_anchors(s,begin,end,cap,seed):
    n=s['n_bins']
    start=max(HISTORY_BINS+1,int(begin)); stop=min(int(end),n-MAX_H-1)
    a=np.arange(start,stop,STRIDE_BINS,dtype=np.int64)
    bad=np.r_[0,np.cumsum(s['near_light'].astype(np.int64))]
    a=a[(bad[a+MAX_H]-bad[a-HISTORY_BINS])==0]
    if len(a)>cap:a=np.sort(np.random.default_rng(seed).choice(a,size=cap,replace=False))
    assert len(a)>=50
    return a
s=sessions['session_9']
# Match Phase4A fold1: fold seed 20261014+31; calibration uses seed +2.
calanchors=sample_anchors(s,.78*s['n_bins'],.90*s['n_bins'],280,SEED+33)
calstd=[];calerr=[]
for a in calanchors:
    pred,spread=frozen_predict(s['rate'][a-HISTORY_BINS:a])
    calstd.append(spread);calerr.append(abs(pred-float(s['rate'][a])))
uncut=max(.01,float(np.quantile(calstd,.90)))
OOD_CUT=float(thresholds.loc[thresholds.fold==1,'history_ood_cut_Hz'].iloc[0])
TARGET=.82*mu
AR_PHI=float(np.clip(alpha*.88,.05,.90))
RHO=.65
GAIN_NOMINAL=-.55*mu
GAIN_MIX=(-.70*mu,+.25*mu)
ACTS=(0.,.5,1.)
from itertools import product
PLANS=list(product(ACTS,repeat=3))
BUDGET=12.0; REFRACTORY_STEPS=2
print('Frozen mu:',round(mu,5),'alpha:',round(alpha,5),'OOD cutoff:',round(OOD_CUT,3),
      'uncertainty cutoff:',round(uncut,5),'target:',round(TARGET,5))
assert GAIN_MIX[0]<0<GAIN_MIX[1] and 0<BUDGET

Frozen mu: 1.87091 alpha: 0.4203 OOD cutoff: 6.491 uncertainty cutoff: 0.07512 target: 1.53414


## 7. Freeze Phase 4A controllers; add a dose-conservative threshold comparator

The original Phase 4A policies retain their logic and objective. `THRESHOLD_HALF` is a **new comparator**, disclosed here. Actions are abstract and dimensionless, not calibrated optical doses.

In [7]:
def candidate_plan(cur, est_next, last_z, prev_action, remaining, gains, worst_case):
    bestcost=np.inf; best_action=0.0
    for plan in PLANS:
        if sum(plan)>remaining+1e-12:continue
        vals=[]
        for gain in gains:
            yp=float(cur);z=float(last_z);prev=float(prev_action);total=0.0
            for i,a in enumerate(plan):
                z=RHO*z+(1-RHO)*a
                natural=est_next if i==0 else mu+alpha*(yp-mu)
                yp=max(0.0,float(natural)+gain*z)
                total+=(max(0.,yp-TARGET)/max(.1,mu))**2+.07*a*a+.02*(a-prev)**2
                prev=float(a)
            vals.append(total)
        score=max(vals) if worst_case else float(np.mean(vals))
        if score<bestcost-1e-10 or (abs(score-bestcost)<1e-10 and plan[0]<best_action):
            bestcost=score;best_action=float(plan[0])
    return best_action

def action_for(policy,hist,t,used,last_action,last_on,z):
    left=BUDGET-used
    if left<0.5-1e-9 or t-last_on<REFRACTORY_STEPS:return 0.,0
    if policy=='NO_LIGHT':return 0.,0
    if policy=='FIXED':return (min(.5,left) if t%7==0 else 0.),0
    h=float(np.mean(hist[-5:]))
    if policy=='THRESHOLD_HALF':return (min(.5,left) if h>TARGET else 0.),0
    if policy=='THRESHOLD':return (min(1.,left) if h>TARGET else 0.),0
    pred,spread=frozen_predict(hist)
    if policy=='MPC_NOMINAL':
        return candidate_plan(hist[-1],pred,z,last_action,left,[GAIN_NOMINAL],False),0
    if policy=='MPC_UNCERTAINTY_GATE':
        if spread>uncut or max(hist[-5:])>OOD_CUT:return 0.,1
        return candidate_plan(hist[-1],pred,z,last_action,left,[GAIN_NOMINAL],False),0
    if policy=='MPC_SIGN_AMBIGUOUS':
        a=candidate_plan(hist[-1],pred,z,last_action,left,GAIN_MIX,True)
        return a,int(a==0)
    raise KeyError(policy)

def hypothetical_gain(scenario,t,n):
    if scenario=='INHIBIT_STRONG':return -.75*mu
    if scenario=='INHIBIT_WEAK':return -.22*mu
    if scenario=='ZERO_EFFECT':return 0.
    if scenario=='EXCITATORY':return +.25*mu
    if scenario=='SIGN_REVERSAL':return -.55*mu if t<n//2 else +.35*mu
    if scenario=='SLOW_ACTUATOR':return -.55*mu
    raise KeyError(scenario)

def actuator_rho(scenario):return .94 if scenario=='SLOW_ACTUATOR' else RHO

## 8. Multi-seed paired counterfactual benchmark (SIMULATION ONLY)

Every policy sees the same initial held-out mouse history and the same seeded disturbance, but **separate simulated state trajectories**. The true optical-response coefficient is **assumed**, including zero and harmful signs. Under the zero-effect scenario all neural trajectories must match exactly across policies.

In [8]:
source_session=sessions['session_12']
# Obtain sufficient nonoverlapping light-free start histories from held-out recording.
if SMOKE_MODE:
    candidate=np.arange(100,source_session['n_bins']-SIM_STEPS-10,250,dtype=np.int64)
else:
    candidate=np.rint(pr[(pr.test_session=='session_12')&(pr.horizon=='0.1s')].t_sec.to_numpy()/BIN_SEC).astype(np.int64)
rng=np.random.default_rng(SEED+88);rng.shuffle(candidate)
starts=[]
for anchor in candidate:
    anchor=int(anchor)
    if anchor<HISTORY_BINS or anchor+SIM_STEPS>=source_session['n_bins']:continue
    if source_session['near_light'][anchor-HISTORY_BINS:anchor+SIM_STEPS].any():continue
    if any(abs(anchor-a)<SIM_STEPS+HISTORY_BINS+10 for a in starts):continue
    starts.append(anchor)
    if len(starts)>=SIM_STARTS:break
assert len(starts)==SIM_STARTS, 'Insufficient independent nonoverlapping held-out start segments'

SIM_ROWS=[]; SIM_TRACES=[]
for start_id,a in enumerate(starts):
    original_hist=source_session['rate'][a-HISTORY_BINS:a].astype(float)
    replay=source_session['rate'][a:a+SIM_STEPS].astype(float)
    for seed_id in range(SIM_SEEDS):
        # Same innovation and shock schedule for every scenario/plant/policy at same start/seed.
        r=np.random.default_rng(SEED+start_id*4729+seed_id*137)
        eps=r.normal(0,.14*mu,SIM_STEPS)
        shock=np.zeros(SIM_STEPS)
        shock[SIM_STEPS//3:SIM_STEPS//3+5]=.65*mu
        for plant in SIM_PLANTS:
            for scenario in SCENARIOS:
                for policy in POLICIES:
                    hist=list(original_hist.copy());last_action=0.;last_on=-999;z=0.;used=0.
                    rr=[];uu=[];abst=[]
                    for t in range(SIM_STEPS):
                        u,ab=action_for(policy,hist,t,used,last_action,last_on,z)
                        u=float(np.clip(u,0,1));assert used+u<=BUDGET+1e-8
                        if u>0:
                            assert t-last_on>=REFRACTORY_STEPS
                            last_on=t
                        used+=u
                        z=actuator_rho(scenario)*z+(1-actuator_rho(scenario))*u
                        gain=hypothetical_gain(scenario,t,SIM_STEPS)
                        if plant=='AR1':
                            natural=mu+AR_PHI*(hist[-1]-mu)+eps[t]+shock[t]
                        else:
                            natural=.85*float(replay[t])+.15*hist[-1]+.25*eps[t]+shock[t]
                        y=float(np.clip(natural+gain*z,0,250))
                        hist.append(y);rr.append(y);uu.append(u);abst.append(ab)
                        if start_id==0 and seed_id==0:
                            SIM_TRACES.append({'plant':plant,'scenario':scenario,'policy':policy,
                                'step':t,'t_sec':t*BIN_SEC,'simulated_rate_Hz':y,
                                'dimensionless_action':u,'ASSUMED_gain_Hz_per_action':gain})
                        last_action=u
                    rr=np.asarray(rr);uu=np.asarray(uu)
                    SIM_ROWS.append({'plant':plant,'scenario':scenario,'policy':policy,
                        'start_id':start_id,'sim_seed_id':seed_id,'N_steps':SIM_STEPS,
                        'mean_simulated_rate_Hz':float(rr.mean()),
                        'mean_excess_above_target_Hz':float(np.maximum(0,rr-TARGET).mean()),
                        'fraction_above_target':float(np.mean(rr>TARGET)),
                        'P90_simulated_rate_Hz':float(np.quantile(rr,.90)),
                        'total_dimensionless_action':float(uu.sum()),
                        'nonzero_action_count':int(np.sum(uu>0)),
                        'abstention_fraction':float(np.mean(abst)),
                        'finite_nonnegative':bool(np.isfinite(rr).all() and (rr>=0).all())})

sim=pd.DataFrame(SIM_ROWS); traces=pd.DataFrame(SIM_TRACES)
base=sim.query('policy=="NO_LIGHT"')[['plant','scenario','start_id','sim_seed_id',
     'mean_simulated_rate_Hz','mean_excess_above_target_Hz']].rename(columns={
     'mean_simulated_rate_Hz':'baseline_NO_LIGHT_rate_Hz',
     'mean_excess_above_target_Hz':'baseline_NO_LIGHT_excess_Hz'})
sim=sim.merge(base,on=['plant','scenario','start_id','sim_seed_id'],validate='many_to_one')
sim['delta_rate_vs_NO_LIGHT_Hz']=sim.mean_simulated_rate_Hz-sim.baseline_NO_LIGHT_rate_Hz
sim['delta_excess_vs_NO_LIGHT_Hz']=sim.mean_excess_above_target_Hz-sim.baseline_NO_LIGHT_excess_Hz
sim['worsened_vs_NO_LIGHT']=sim.delta_rate_vs_NO_LIGHT_Hz>1e-8
assert sim.finite_nonnegative.all()
assert sim.total_dimensionless_action.le(BUDGET+1e-8).all()
assert sim.groupby(['plant','scenario','start_id','sim_seed_id']).policy.nunique().eq(len(POLICIES)).all()
# Zero gain must eliminate EVERY effect on the plant, regardless of policy action.
zero=sim.query('scenario=="ZERO_EFFECT"')
assert zero.groupby(['plant','start_id','sim_seed_id']).mean_simulated_rate_Hz.agg(lambda q:q.max()-q.min()).max()<1e-10
assert (sim.query('policy=="NO_LIGHT"').total_dimensionless_action==0).all()
sim.to_csv(OUT/'SIMULATION_ONLY_Table4_All_Paired_Episodes.csv',index=False)
traces.to_csv(OUT/'SIMULATION_ONLY_Representative_Trajectories.csv',index=False)
print('Completed',len(sim),'paired synthetic episodes (not biological replicates)')
print('Zero-effect invariance VERIFIED')

Completed 1512 paired synthetic episodes (not biological replicates)
Zero-effect invariance VERIFIED


## 9. Predeclared descriptive statistics: simulation trade-offs, zero effect, harmful signs

Hierarchical descriptive intervals resample initial recorded histories and seeded disturbances; they **do not** represent independent-animal confidence intervals and should not be used for biological p-values. Compare all controllers against NO_LIGHT and THRESHOLD as paired descriptive comparisons.

In [9]:
def hierarchical_descriptive_ci(df,col,seed):
    groups={k:g[col].to_numpy(float) for k,g in df.groupby('start_id')}
    keys=list(groups)
    rng=np.random.default_rng(seed)
    vals=[]
    for _ in range(BOOT_REPS):
        picks=rng.choice(keys,size=len(keys),replace=True)
        numbers=[]
        for key in picks:
            vv=groups[key]
            numbers.extend(rng.choice(vv,size=len(vv),replace=True))
        vals.append(np.mean(numbers))
    return np.quantile(vals,[.025,.975])

SUMMARY=[]
for (plant,scenario,policy),g in sim.groupby(['plant','scenario','policy'],sort=True):
    lo,hi=hierarchical_descriptive_ci(g,'delta_rate_vs_NO_LIGHT_Hz',
        SEED+len(policy)*7+len(scenario)*5+len(plant))
    SUMMARY.append({'plant':plant,'scenario':scenario,'policy':policy,
      'n_synthetic_episodes':len(g),'n_recording_start_histories':g.start_id.nunique(),
      'n_simulated_seeds_per_start':g.sim_seed_id.nunique(),
      'mean_simulated_rate_Hz':g.mean_simulated_rate_Hz.mean(),
      'mean_excess_above_target_Hz':g.mean_excess_above_target_Hz.mean(),
      'mean_total_dimensionless_action':g.total_dimensionless_action.mean(),
      'mean_delta_rate_vs_NO_LIGHT_Hz':g.delta_rate_vs_NO_LIGHT_Hz.mean(),
      'DESCRIPTIVE_CI95_low_Hz_NOT_BIOLOGICAL':lo,
      'DESCRIPTIVE_CI95_high_Hz_NOT_BIOLOGICAL':hi,
      'fraction_harmful_episodes_vs_NO_LIGHT':g.worsened_vs_NO_LIGHT.mean(),
      'mean_abstention_fraction':g.abstention_fraction.mean()})
summary=pd.DataFrame(SUMMARY)
summary['pareto_dominated_in_plant_scenario']=False
for (plant,scenario),grp in summary.groupby(['plant','scenario']):
    for row in grp.itertuples():
        other=grp[(grp.policy!=row.policy)&
              (grp.mean_excess_above_target_Hz<=row.mean_excess_above_target_Hz+1e-9)&
              (grp.mean_total_dimensionless_action<=row.mean_total_dimensionless_action+1e-9)&
              ((grp.mean_excess_above_target_Hz<row.mean_excess_above_target_Hz-1e-9)|
               (grp.mean_total_dimensionless_action<row.mean_total_dimensionless_action-1e-9))]
        summary.loc[row.Index,'pareto_dominated_in_plant_scenario']=bool(len(other))
summary.to_csv(OUT/'SIMULATION_ONLY_Table5_Policy_Stress_Summary.csv',index=False)
print(summary.query('plant=="AR1" and scenario in ["INHIBIT_STRONG","EXCITATORY","ZERO_EFFECT"]')
      [['scenario','policy','mean_delta_rate_vs_NO_LIGHT_Hz','mean_total_dimensionless_action',
        'fraction_harmful_episodes_vs_NO_LIGHT']].round(3).to_string(index=False))

      scenario               policy  mean_delta_rate_vs_NO_LIGHT_Hz  mean_total_dimensionless_action  fraction_harmful_episodes_vs_NO_LIGHT
    EXCITATORY                FIXED                           0.052                            2.500                                    1.0
    EXCITATORY          MPC_NOMINAL                           0.172                            8.750                                    1.0
    EXCITATORY   MPC_SIGN_AMBIGUOUS                           0.000                            0.000                                    0.0
    EXCITATORY MPC_UNCERTAINTY_GATE                           0.121                            6.306                                    1.0
    EXCITATORY             NO_LIGHT                           0.000                            0.000                                    0.0
    EXCITATORY            THRESHOLD                           0.253                           12.000                                    1.0
    EXCITATORY      

## 10. Publication figures: observed versus hypothetical kept visually distinct

Every caption has evidence type in the title. Figures have actual Phase 4A real predictions or Phase 4B explicitly simulated outputs. Export at 250 dpi; no synthetic predictions are displayed as observed neuroscience.

In [10]:
plt.rcParams.update({'font.size':9,'axes.titlesize':11,'figure.dpi':120})
fig,ax=plt.subplots(figsize=(9,4.3))
for sess,g in actual.groupby('heldout_session'):
    gg=g[g.model=='GRU_uniform'].set_index('horizon').reindex(HORIZON_NAMES)
    ax.plot(HORIZON_NAMES,gg.MAE_Hz.to_numpy(),marker='o',label=sess+' GRU')
ax.set(xlabel='Prediction horizon',ylabel='MAE (Hz per neural unit)',
       title='REAL OBSERVED: Frozen Phase 4A held-out neural forecasting')
ax.grid(alpha=.2);ax.legend();fig.tight_layout()
fig.savefig(OUT/'fig1_REAL_frozen_GRU_forecasting.png',dpi=250);plt.close(fig)

fig,ax=plt.subplots(figsize=(8.5,4.5))
cc=ci[(ci.model=='GRU_uniform')&(ci.horizon=='6s')&
      (ci.interval_method=='SYMMETRIC')&(ci.group.isin(['ALL','HIGH_TRUE_TARGET_REVIEW_ONLY']))]
labels=[];values=[]
for sess in ['session_12','session_9']:
    for group in ['ALL','HIGH_TRUE_TARGET_REVIEW_ONLY']:
        q=cc[(cc.session==sess)&(cc.group==group)]
        assert len(q)==1
        labels.append(sess.replace('session_','S')+' '+('All' if group=='ALL' else 'High target'))
        values.append(float(q.coverage.iloc[0])*100)
ax.bar(labels,values);ax.axhline(90,ls='--',lw=1,label='90% nominal')
ax.set(ylabel='Empirical coverage (%)',ylim=(0,105),title='REAL OBSERVED: 6s prediction-interval reliability')
ax.tick_params(axis='x',labelrotation=25);ax.legend();fig.tight_layout()
fig.savefig(OUT/'fig2_REAL_uncertainty_limitation.png',dpi=250);plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(12,4.5),sharey=True)
for ax,scenario in zip(axs,['INHIBIT_STRONG','EXCITATORY']):
    gg=summary[(summary.plant=='AR1')&(summary.scenario==scenario)].set_index('policy').reindex(POLICIES)
    ax.bar(range(len(gg)),gg.mean_delta_rate_vs_NO_LIGHT_Hz)
    ax.axhline(0,lw=1);ax.set_xticks(range(len(gg)))
    ax.set_xticklabels([p.replace('MPC_','MPC ').replace('THRESHOLD','THR') for p in POLICIES],rotation=65,ha='right')
    ax.set(title=scenario,ylabel='Delta vs no light (Hz)' if ax==axs[0] else None)
fig.suptitle('SIMULATION ONLY: Policy effects under assumed optical gains')
fig.tight_layout()
fig.savefig(OUT/'fig3_SIMULATION_ONLY_gain_sign_sensitivity.png',dpi=250);plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(11,4.3))
for ax,scenario in zip(axs,['INHIBIT_STRONG','EXCITATORY']):
    gg=summary[(summary.plant=='REPLAY')&(summary.scenario==scenario)]
    for r in gg.itertuples():
        ax.scatter(r.mean_total_dimensionless_action,r.mean_excess_above_target_Hz,s=45)
        ax.annotate(r.policy.replace('MPC_',''),(r.mean_total_dimensionless_action,
                r.mean_excess_above_target_Hz),fontsize=7,xytext=(2,3),textcoords='offset points')
    ax.set(title=scenario,xlabel='Mean dimensionless action',ylabel='Simulated mean excess above target (Hz)')
fig.suptitle('SIMULATION ONLY: Activity-action trade-offs (not biological efficacy)')
fig.tight_layout()
fig.savefig(OUT/'fig4_SIMULATION_ONLY_policy_pareto.png',dpi=250);plt.close(fig)
print('Saved two REAL and two SIMULATION-ONLY figures.')

Saved two REAL and two SIMULATION-ONLY figures.


## 11. Evidence gates, defensible manuscript draft, and final ZIP

The manuscript text is automatically generated **only from computed figures/tables**. The journal's Q1 ranking cannot be inferred from computational correctness. Read the limitations before using these paragraphs in a submission.

In [11]:
EVIDENCE=[]
def gate(name,status,detail):EVIDENCE.append({'criterion':name,'status':status,'evidence':detail})
gate('Frozen model reproducibility','PASS' if not SMOKE_MODE else 'NOT TESTED',
     'Checkpoint predictions verified against archived Phase4A rows; all 16 metric groups independently recalculated')
gate('Independent-animal validation','NOT ESTABLISHED','One mouse; session-held-out only')
gate('High-activity six-second uncertainty calibration','LIMITATION',
     'Report high-target coverage from REAL_Table2; heldout groups under nominal coverage')
gate('Experimental light-response model','NOT ESTABLISHED',
     'Simulated action gains and actuator dynamics are assumptions; not estimated from synchronized optical intervention trials')
gate('MPC superiority over threshold controller','CONDITIONAL SIMULATION ONLY',
     'Compare synthetic rate and action Pareto results by plant and optical-effect assumption')
gate('Unexpected excitatory sign robustness','LIMITATION',
     'Inspect EXCITATORY and SIGN_REVERSAL harm fractions; default abstaining controller is no-stimulation equivalent')
gate('Human epilepsy benefit or clinical safety','NOT ESTABLISHED',
     'No human clinical data and no biologically validated intervention')
pd.DataFrame(EVIDENCE).to_csv(OUT/'TABLE6_Evidence_Gates_and_Research_Limitations.csv',index=False)

row=(actual[(actual.heldout_session=='session_12')&(actual.horizon=='0.1s')&(actual.model=='GRU_uniform')].iloc[0])
row2=(actual[(actual.heldout_session=='session_9')&(actual.horizon=='0.1s')&(actual.model=='GRU_uniform')].iloc[0])
a=summary.query('plant=="AR1" and scenario=="INHIBIT_STRONG" and policy=="THRESHOLD"').iloc[0]
b=summary.query('plant=="AR1" and scenario=="INHIBIT_STRONG" and policy=="MPC_NOMINAL"').iloc[0]
c=summary.query('plant=="AR1" and scenario=="EXCITATORY" and policy=="MPC_NOMINAL"').iloc[0]

manuscript=f"""# Phase 4B | Results and Discussion (evidence-restricted draft)

## Study design and evidence separation

We conducted a frozen-model computational sensitivity benchmark for a digital-twin-inspired neural control framework. The real-data component uses mouse hippocampal spike activity from two sessions in DANDI {DANDISET}, version {VERSION}. Phase 4A GRU weights and preprocessing are reused without retraining. All optogenetic control experiments are synthetic and use assumed action gains; the simulation does not establish biological stimulation effects.

## Real observed neural-activity forecasting

On the session-12 held-out recording, the frozen standard GRU achieved 0.1-s MAE of {row.MAE_Hz:.4f} Hz per neural unit (R²={row.R2:.3f}); on session 9 the corresponding MAE was {row2.MAE_Hz:.4f} Hz (R²={row2.R2:.3f}). Both recordings are from one animal and thus do not establish biological generalization. The full multi-horizon results and Phase 4A uncertainty diagnostics are in REAL_Table1 and REAL_Table2. Poor high-target coverage and negative mean R² for six-second free-running segments remain limitations.

## Simulation-only comparative controller evaluation

The model-predictive-control policies were evaluated under {len(SCENARIOS)} specified hypothetical light-effect scenarios and {len(SIM_PLANTS)} distinct toy plant equations, each with {SIM_SEEDS} numerical seeds and {SIM_STARTS} held-out starting histories. For the AR1 plant under assumed strong inhibition, THRESHOLD produced mean simulated rate change vs no light of {a.mean_delta_rate_vs_NO_LIGHT_Hz:+.4f} Hz at mean action {a.mean_total_dimensionless_action:.3f}; nominal MPC produced {b.mean_delta_rate_vs_NO_LIGHT_Hz:+.4f} Hz at mean action {b.mean_total_dimensionless_action:.3f}. Under an excitatory assumed gain, nominal MPC's mean change vs no light was {c.mean_delta_rate_vs_NO_LIGHT_Hz:+.4f} Hz, so harmful model misspecification remains possible. Repeated synthetic seeds do not increase the number of independent animals.

## Interpretation

This is a computational robustness and failure-mode analysis, not proof of effective personalized neuromodulation. Policy rankings depend on the assumed plant, gains, objective weights, and action budgets. Any threshold-controller advantage and any MPC action savings must be reported together; there is no evidence of a universally superior policy. A sign-ambiguity controller that uniformly abstains is effectively equivalent to no stimulation rather than demonstrating informed therapeutic action.

## Limitations

Only one mouse was available for the frozen real-data component; independent-animal generalization, human epilepsy outcomes, and experimentally measured stimulation-dose responses are absent. Forecasting errors remain substantial at high activity and over multi-step free-running horizons. Nominal 90% intervals are unreliable under high-target activity and session shift. Stimulation gains, response signs, light doses, dynamics, and controller targets are assumed and dimensionless, not experimentally calibrated. Numerical replicates are not independent biological samples. No clinical safety, epilepsy suppression, optogenetic therapeutic efficacy, or Q1-level novelty is established by this computational benchmark alone.

## Additional evidence needed

A future biological study requires synchronized calibrated stimulation-response trials, independent animals (and human pathological tissue for human-focused claims), improved high-activity calibration, and validation against relevant biological intervention controls. Before selecting a Q1 journal, perform a fresh literature comparison to establish a measurable methodological novelty beyond existing hybrid digital twins and model-predictive control.
"""
(OUT/'Phase4B_Results_Discussion_Manuscript_DRAFT.md').write_text(manuscript)

provenance={'phase':'4B','data_source':'Phase4A frozen experiment outputs',
    'dataset':DANDISET,'version':VERSION,'independent_animals':1,
    'model_checkpoints_frozen':True,'train_models_this_phase':False,
    'sha256_frozen_bundle':EXPECTED_BUNDLE_SHA256,
    'quick_mode':QUICK_MODE,'simulated_seed_count':SIM_SEEDS,
    'starting_histories':SIM_STARTS,'sim_step_count':SIM_STEPS,
    'plants':SIM_PLANTS,'scenarios':SCENARIOS,'policies':POLICIES,
    'unit_of_action':'dimensionless; no physical light calibration',
    'synthetic_not_for_journal':SMOKE_MODE,
    'methodology_warning':'All control effects hypothetical; descriptive bootstrap over synthetic noise cannot establish biological inference',
    'python':sys.version,'torch':torch.__version__,'numpy':np.__version__,'pandas':pd.__version__}
(OUT/'PROVENANCE_Phase4B.json').write_text(json.dumps(provenance,indent=2))
(OUT/'README_FIRST.txt').write_text('PHASE 4B: REAL_* = frozen Phase4A mouse neural predictions; SIMULATION_ONLY_* = hypothetical toy controller experiments. One independent mouse. Synthetic seeds are not animals. No human epilepsy or biological stimulation efficacy claims. Consult Results_Discussion draft and evidence gates.\n')

archive=ROOT/'phase4b_final_computational_evaluation_results.zip'
if SMOKE_MODE:
    archive=ROOT/'phase4b_SYNTHETIC_SMOKE_TEST_DO_NOT_REPORT.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for file in sorted(OUT.iterdir()):
        if file.is_file():z.write(file,file.name)
assert archive.exists() and archive.stat().st_size>1000
print('FINISHED:',archive,'MB:',round(archive.stat().st_size/1e6,2))
print('Evidence: real forecasting is from Phase4A, only simulation expanded in Phase4B.')

FINISHED: /kaggle/working/phase4b_final_computational_evaluation_results.zip MB: 0.79
Evidence: real forecasting is from Phase4A, only simulation expanded in Phase4B.


## After the run

1. In Kaggle **Output** / `/kaggle/working/`, download `phase4b_final_computational_evaluation_results.zip`.
2. Share it for independent verification, including `REAL_Table1_Frozen_Heldout_Regression.csv`, `SIMULATION_ONLY_Table5_Policy_Stress_Summary.csv`, and the generated manuscript draft.
3. **Never report synthetic smoke-test output or hypothetical stimulation as validated optogenetic treatment.** A Q1 manuscript still needs a defined novelty claim and ideally independent biological validation.